De identification of DICOM files

In [1]:
!pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.5 MB/s eta 0:00:00


In [2]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 9.0 MB/s eta 0:00:00


In [3]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 70.4 MB/s eta 0:00:00


In [4]:
import os
import fitz
import re
from pypdf import PdfReader, PdfWriter

In [5]:
import os

DICOM_OUTPUT = "/content/drive/MyDrive/Origin_Output/DICOMs"
PDF_OUTPUT = "/content/drive/MyDrive/Origin_Output/PDFs"

DICOM_INPUT = "/content/drive/MyDrive/RC-datasets/RC1/assets/DICOMs"
PDF_INPUT = "/content/drive/MyDrive/RC-datasets/RC1/assets/PDFs"

os.makedirs(DICOM_OUTPUT, exist_ok=True)
os.makedirs(PDF_OUTPUT, exist_ok=True)

print("Folders configured successfully")

Folders configured successfully


In [6]:

dicom_files = []

for file in os.listdir(DICOM_INPUT):
    path = os.path.join(DICOM_INPUT, file)

    if os.path.isfile(path):
        dicom_files.append(path)

print("Total DICOM files:", len(dicom_files))

Total DICOM files: 480


In [7]:
pdf_files = []

for file in os.listdir(PDF_INPUT):
    path = os.path.join(PDF_INPUT, file)

    if os.path.isfile(path):
        pdf_files.append(path)

print("Total PDF files:", len(pdf_files))

Total PDF files: 50


In [8]:
print("First 10 DICOM files:")
for file in dicom_files[:10]:
    print(os.path.basename(file))

print("\nFirst 10 PDF files:")
for file in pdf_files[:10]:
    print(os.path.basename(file))

First 10 DICOM files:
6415974217_06-09-1988-ABDOMENPELVIS-29078_237_000000-PJN-15958_1-04.dcm
6670427471_05-26-2000-FORFILE_CT_ABD_ANDOR_PEL_-_CD-25398_5_000000-NEPHRO__4_0__B40f__M0_4-18678_1-104.dcm
571403367_07-11-2019-DBT_Reconstructed_Volume-37558_DBT_slices-78838_52-01.dcm
571403367_07-11-2019-DBT_Reconstructed_Volume-37558_DBT_slices-78838_28-01.dcm
6670427471_05-26-2000-FORFILE_CT_ABD_ANDOR_PEL_-_CD-25398_5_000000-NEPHRO__4_0__B40f__M0_4-18678_1-029.dcm
8732322741_01-05-2008-MRI_PROSTATE_W_WO_CONTRAST-39318_4_000000-t2spcrstaxial_oblProstate-50358_1-01.dcm
571403367_07-11-2019-DBT_Reconstructed_Volume-37558_DBT_slices-78838_45-01.dcm
3209648408_09-23-1999-CT_UROGRAM-31798_3_000000-PARENCHYMAL_PHASE_Sep1999-95798_1-109.dcm
3209648408_09-23-1999-CT_UROGRAM-31798_3_000000-PARENCHYMAL_PHASE_Sep1999-95798_1-053.dcm
571403367_07-11-2019-DBT_Reconstructed_Volume-37558_DBT_slices-78838_37-01.dcm

First 10 PDF files:
patient_14392.pdf
patient_84709.pdf
patient_59680.pdf
patient_16839.pd

Working on one file first

In [9]:
import pydicom

sample_file = dicom_files[0]

dicom = pydicom.dcmread(sample_file)

print("DICOM loaded successfully")


DICOM loaded successfully


In [10]:
import hmac
import hashlib

#SALT AND HASH
from google.colab import userdata
SALT = userdata.get('DEID_SALT').encode("utf-8")

def hmac_hex(value: str) -> str:
    return hmac.new(SALT, value.encode("utf-8"), hashlib.sha256).hexdigest()

In [11]:
original_patient_id = dicom.PatientID
pseudo_patient_id = "ANON" + hmac_hex(str(original_patient_id))[:16].upper()

dicom.PatientID = pseudo_patient_id

print("Original:", original_patient_id)
print("Pseudonymous:", pseudo_patient_id)

Original: 6415974217
Pseudonymous: ANON0CBAE706C89F5103


In [12]:
#check
d1 = pydicom.dcmread(dicom_files[0])
d2 = pydicom.dcmread(dicom_files[1])
print(hmac_hex(str(d1.PatientID)) == hmac_hex(str(d2.PatientID)))

False


In [13]:
dicom.PatientName = "ANONYMOUS"
print(dicom.PatientName)

ANONYMOUS


In [14]:
def get_date_offset(patient_id: str) -> int:
    digest = hmac_hex(str(patient_id))
    raw_int = int(digest[:8], 16)   # first 8 hex chars from a patient_id, convert to a number for every date field belonging to this patient.
    offset = (raw_int % 729) - 364  # squash into range (-364,364)
    return offset

offset_days = get_date_offset(original_patient_id)
print("This patient's date offset:", offset_days, "days")

This patient's date offset: -64 days


In [15]:
from datetime import datetime, timedelta

if "PatientBirthDate" in dicom:
    raw_date = str(dicom.PatientBirthDate)   #"YYYYMMDD"

    if raw_date:
        parsed_date = datetime.strptime(raw_date, "%Y%m%d")
        shifted_date = parsed_date + timedelta(days=offset_days)
        dicom.PatientBirthDate = shifted_date.strftime("%Y%m%d")

print("Shifted birth date:", dicom.get("PatientBirthDate"))

Shifted birth date: 19370228


In [16]:
# not needed fields for analysis
remove_fields = [
    "PatientAddress",
    "PatientTelephoneNumbers",
    "ReferringPhysicianName",
    "PerformingPhysicianName",
    "InstitutionName",
    "InstitutionAddress",
    "OperatorsName",
    "StationName",
    "OtherPatientIDs",
    "OtherPatientNames",
    "AccessionNumber",
    "StudyID",
]

removed_count = 0
for field in remove_fields:
    if field in dicom:
        del dicom[field]
        removed_count += 1

print(f"Removed {removed_count} of {len(remove_fields)} fields (rest weren't present in this file)")

Removed 8 of 12 fields (rest weren't present in this file)


In [17]:
#sanity check
for field in remove_fields:
    print(field, ":", "still present" if field in dicom else "removed or absent")

PatientAddress : removed or absent
PatientTelephoneNumbers : removed or absent
ReferringPhysicianName : removed or absent
PerformingPhysicianName : removed or absent
InstitutionName : removed or absent
InstitutionAddress : removed or absent
OperatorsName : removed or absent
StationName : removed or absent
OtherPatientIDs : removed or absent
OtherPatientNames : removed or absent
AccessionNumber : removed or absent
StudyID : removed or absent


In [18]:
before_count = len(dicom)
dicom.remove_private_tags()
after_count = len(dicom)

print(f"Tag count before: {before_count}")
print(f"Tag count after: {after_count}")
print(f"Private tags removed: {before_count - after_count}")

Tag count before: 284
Tag count after: 87
Private tags removed: 197


In [19]:
def check_burned_in_risk(dicom):
    modality = str(dicom.get("Modality", ""))
    burned_in_flag = str(dicom.get("BurnedInAnnotation", "")).upper()

    risky_modalities = ["US", "OT", "SC"]  # ultrasound, other, secondary capture

    if burned_in_flag == "YES":
        return True, "BurnedInAnnotation tag explicitly set to YES"
    if modality in risky_modalities:
        return True, f"modality '{modality}' has known burned-in-PHI risk, flag unset/unreliable"

    return False, None

Final Function including all sanity checks

In [20]:
def deidentify_dicom(dicom):
    original_patient_id = dicom.PatientID if "PatientID" in dicom else "UNKNOWN"

    hashed_count = 0
    date_shifted_count = 0
    removed_count = 0

    # pseudonymous PatientID
    pseudo_patient_id = "ANON" + hmac_hex(str(original_patient_id))[:16].upper()
    dicom.PatientID = pseudo_patient_id
    hashed_count += 1

    # hash UIDs
    uid_fields = ["StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID", "FrameOfReferenceUID"]
    for uid_field in uid_fields:
        if uid_field in dicom:
            original_uid = str(dicom.get(uid_field))
            digest = hmac_hex(original_uid)
            new_uid = f"2.25.{int(digest, 16)}"[:64]
            setattr(dicom, uid_field, new_uid)
            hashed_count += 1

    # PatientName
    dicom.PatientName = "ANONYMOUS"

    # shift dates
    offset_days = get_date_offset(original_patient_id)
    date_fields = ["PatientBirthDate", "StudyDate", "SeriesDate"]
    for date_field in date_fields:
        if date_field in dicom:
            raw_date = str(dicom.get(date_field))
            if raw_date:
                try:
                    parsed_date = datetime.strptime(raw_date, "%Y%m%d")
                    shifted_date = parsed_date + timedelta(days=offset_days)
                    setattr(dicom, date_field, shifted_date.strftime("%Y%m%d"))
                    date_shifted_count += 1
                except ValueError:
                    setattr(dicom, date_field, "")

    # remove direct-identifier
    def scrub_dataset(ds):
        nonlocal removed_count
        for field in remove_fields:
            if field in ds:
                del ds[field]
                removed_count += 1
        for elem in ds:
            if elem.VR == "SQ":
                for item in elem.value:
                    scrub_dataset(item)

    scrub_dataset(dicom)

    # strip private tags
    dicom.remove_private_tags()

    return pseudo_patient_id, removed_count, hashed_count, date_shifted_count

In [21]:
import hashlib

def get_file_hash(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

def make_output_filename(pseudo_id: str, original_path: str) -> str:
    path_hash = hashlib.sha256(original_path.encode("utf-8")).hexdigest()[:10]
    ext = os.path.splitext(original_path)[1]
    return f"{pseudo_id}_{path_hash}{ext}"

Setting up the metadata database

In [22]:
!apt-get -y install postgresql > /dev/null
!service postgresql start

!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'deidpipeline';"
!sudo -u postgres psql -c "CREATE DATABASE deid_pipeline;"

!pip install psycopg2-binary -q

 * Starting PostgreSQL 14 database server
   ...done.
ALTER ROLE
CREATE DATABASE
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 56.5 MB/s eta 0:00:00


In [23]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    dbname="deid_pipeline",
    user="postgres",
    password="deidpipeline"
)
conn.autocommit = True
cur = conn.cursor()

print("Connected:", conn.status == psycopg2.extensions.STATUS_READY)

Connected: True


In [24]:
cur.execute("""
CREATE TABLE IF NOT EXISTS patients (
    pseudo_patient_id   TEXT PRIMARY KEY,
    sex                 TEXT,
    age                 TEXT,
    first_seen_at       TIMESTAMP DEFAULT NOW(),
    last_seen_at        TIMESTAMP DEFAULT NOW()
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS dicom_files (
    file_id                 SERIAL PRIMARY KEY,
    pseudo_patient_id       TEXT REFERENCES patients(pseudo_patient_id),
    source_file_hash        TEXT UNIQUE NOT NULL,
    output_filename          TEXT NOT NULL,
    modality                TEXT,
    manufacturer             TEXT,
    manufacturer_model       TEXT,
    body_part_examined       TEXT,
    rows_px                  INTEGER,
    columns_px                INTEGER,
    pixel_spacing            TEXT,
    tags_removed             INTEGER,
    tags_hashed               INTEGER,
    tags_date_shifted         INTEGER,
    quarantined               BOOLEAN DEFAULT FALSE,
    processed_at              TIMESTAMP DEFAULT NOW()
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS pdf_reports (
    file_id                 SERIAL PRIMARY KEY,
    pseudo_patient_id       TEXT REFERENCES patients(pseudo_patient_id),
    source_file_hash        TEXT UNIQUE NOT NULL,
    output_filename          TEXT NOT NULL,
    redaction_verified       BOOLEAN,
    processed_at              TIMESTAMP DEFAULT NOW()
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS pipeline_runs (
    run_id               SERIAL PRIMARY KEY,
    run_type             TEXT NOT NULL,   -- 'dicom' or 'pdf'
    started_at            TIMESTAMP,
    completed_at           TIMESTAMP,
    files_processed        INTEGER DEFAULT 0,
    files_skipped           INTEGER DEFAULT 0,
    files_quarantined       INTEGER DEFAULT 0,
    files_failed             INTEGER DEFAULT 0
);
""")

print("Tables created")

Tables created


In [25]:
cur.execute("""
SELECT table_name FROM information_schema.tables
WHERE table_schema = 'public';
""")
print(cur.fetchall())

[('patients',), ('dicom_files',), ('pdf_reports',), ('pipeline_runs',)]


In [26]:
def upsert_patient(pseudo_patient_id: str, sex: str = None, age: str = None):

    cur.execute("""
        INSERT INTO patients (pseudo_patient_id, sex, age, first_seen_at, last_seen_at)
        VALUES (%s, %s, %s, NOW(), NOW())
        ON CONFLICT (pseudo_patient_id) DO UPDATE
        SET last_seen_at = NOW();
    """, (pseudo_patient_id, sex, age))


def insert_dicom_record(record: dict):
    cur.execute("""
        INSERT INTO dicom_files (
            pseudo_patient_id, source_file_hash, output_filename,
            modality, manufacturer, manufacturer_model, body_part_examined,
            rows_px, columns_px, pixel_spacing,
            tags_removed, tags_hashed, tags_date_shifted, quarantined
        ) VALUES (
            %(pseudo_patient_id)s, %(source_file_hash)s, %(output_filename)s,
            %(modality)s, %(manufacturer)s, %(manufacturer_model)s, %(body_part_examined)s,
            %(rows_px)s, %(columns_px)s, %(pixel_spacing)s,
            %(tags_removed)s, %(tags_hashed)s, %(tags_date_shifted)s, %(quarantined)s
        )
        ON CONFLICT (source_file_hash) DO NOTHING;
    """, record)


def insert_pdf_record(record: dict):
    cur.execute("""
        INSERT INTO pdf_reports (
            pseudo_patient_id, source_file_hash, output_filename, redaction_verified
        ) VALUES (
            %(pseudo_patient_id)s, %(source_file_hash)s, %(output_filename)s, %(redaction_verified)s
        )
        ON CONFLICT (source_file_hash) DO NOTHING;
    """, record)

Processing and saving all de-identified dicom files

In [27]:
import shutil
from datetime import datetime

QUARANTINE_DIR = "/content/drive/MyDrive/Origin_Output/Quarantine_DICOMs"
os.makedirs(QUARANTINE_DIR, exist_ok=True)

def is_already_processed_dicom(file_hash: str) -> bool:
    cur.execute("SELECT 1 FROM dicom_files WHERE source_file_hash = %s", (file_hash,))
    return cur.fetchone() is not None

summary = {"processed": 0, "skipped": 0, "quarantined": 0, "failed": 0}
failed_files = []
run_started = datetime.now()

for path in dicom_files:
    try:
        file_hash = get_file_hash(path)

        if is_already_processed_dicom(file_hash):
            summary["skipped"] += 1
            continue

        dicom = pydicom.dcmread(path)

        is_risky, reason = check_burned_in_risk(dicom)
        if is_risky:
            shutil.copy2(path, os.path.join(QUARANTINE_DIR, os.path.basename(path)))
            summary["quarantined"] += 1
            continue

        pseudo_id, removed_count, hashed_count, date_shifted_count = deidentify_dicom(dicom)

        out_filename = make_output_filename(pseudo_id, path)
        out_path = os.path.join(DICOM_OUTPUT, out_filename)
        dicom.save_as(out_path)

        upsert_patient(pseudo_id, sex=str(dicom.get("PatientSex", "")), age=str(dicom.get("PatientAge", "")))

        insert_dicom_record({
            "pseudo_patient_id": pseudo_id,
            "source_file_hash": file_hash,
            "output_filename": out_filename,
            "modality": str(dicom.get("Modality", "")),
            "manufacturer": str(dicom.get("Manufacturer", "")),
            "manufacturer_model": str(dicom.get("ManufacturerModelName", "")),
            "body_part_examined": str(dicom.get("BodyPartExamined", "")),
            "rows_px": int(dicom.get("Rows", 0)) or None,
            "columns_px": int(dicom.get("Columns", 0)) or None,
            "pixel_spacing": str(dicom.get("PixelSpacing", "")),
            "tags_removed": removed_count,
            "tags_hashed": hashed_count,
            "tags_date_shifted": date_shifted_count,
            "quarantined": False,
        })

        summary["processed"] += 1
    except Exception as e:
        summary["failed"] += 1
        failed_files.append((os.path.basename(path), str(e)))

cur.execute("""
    INSERT INTO pipeline_runs (run_type, started_at, completed_at, files_processed, files_skipped, files_quarantined, files_failed)
    VALUES (\'dicom\', %s, NOW(), %s, %s, %s, %s)
""", (run_started, summary["processed"], summary["skipped"], summary["quarantined"], summary["failed"]))

print("=== DICOM batch complete ===")
print(summary)
print(f"Total accounted for: {sum(summary.values())} / {len(dicom_files)}")
if failed_files:
    print("\nFailed files:")
    for name, error in failed_files:
        print(f"  {name}: {error}")

=== DICOM batch complete ===
{'processed': 480, 'skipped': 0, 'quarantined': 0, 'failed': 0}
Total accounted for: 480 / 480


Checking the saved dicom outputs

In [28]:

output_files = os.listdir(DICOM_OUTPUT)

import random
sample_outputs = random.sample(output_files, 7)
for fname in sample_outputs:
    d = pydicom.dcmread(os.path.join(DICOM_OUTPUT, fname))
    print(f"\n{fname}")
    print(f"  PatientName: {d.get('PatientName')}")
    print(f"  PatientID: {d.get('PatientID')}")
    print(f"  PatientBirthDate: {d.get('PatientBirthDate', 'not present')}")
    print(f"  Modality: {d.get('Modality')}, Manufacturer: {d.get('Manufacturer')}")

# Check for Duplicate Patient IDS
from collections import Counter
pseudo_ids = [f.split("_")[0] for f in output_files]
id_counts = Counter(pseudo_ids)
repeats = {pid: count for pid, count in id_counts.items() if count > 1}

# Result
print(f"\n{len(repeats)} pseudonymous patient IDs appear in more than one file "
      f"(i.e. {sum(repeats.values())} files belong to {len(repeats)} repeat patients)")


ANON13FA77C410E76BF7_d6df5edfcb.dcm
  PatientName: ANONYMOUS
  PatientID: ANON13FA77C410E76BF7
  PatientBirthDate: 19330622
  Modality: CT, Manufacturer: GE MEDICAL SYSTEMS

ANOND25531C9226757BF_2a1bba2b60.dcm
  PatientName: ANONYMOUS
  PatientID: ANOND25531C9226757BF
  PatientBirthDate: 19240919
  Modality: CT, Manufacturer: SIEMENS

ANON13FA77C410E76BF7_175efbc097.dcm
  PatientName: ANONYMOUS
  PatientID: ANON13FA77C410E76BF7
  PatientBirthDate: 19330622
  Modality: CT, Manufacturer: GE MEDICAL SYSTEMS

ANON13FA77C410E76BF7_ffee98949a.dcm
  PatientName: ANONYMOUS
  PatientID: ANON13FA77C410E76BF7
  PatientBirthDate: 19330622
  Modality: CT, Manufacturer: GE MEDICAL SYSTEMS

ANONEC85130877A0A081_ddc50b5660.dcm
  PatientName: ANONYMOUS
  PatientID: ANONEC85130877A0A081
  PatientBirthDate: 19440922
  Modality: MR, Manufacturer: SIEMENS

ANOND25531C9226757BF_e9b3068fb2.dcm
  PatientName: ANONYMOUS
  PatientID: ANOND25531C9226757BF
  PatientBirthDate: 19240919
  Modality: CT, Manufacture

De-Identification of PDFs

In [29]:
sample_pdf = pdf_files[0]

reader = PdfReader(sample_pdf)
print(f"File: {os.path.basename(sample_pdf)}")
print(f"Pages: {len(reader.pages)}")
print("\n--- Extracted text from page 1 ---\n")
print(reader.pages[0].extract_text())

File: patient_14392.pdf
Pages: 1

--- Extracted text from page 1 ---

Origin Hospital
Patient ID :
Patient Name :
Gender : Female
Examination Findings
Patient Age :
GA :
BMI :
Head :
Brain :
Heart :
Spine :
Abdominal wall:
Urinary tract:
Extremities:
Conclusion
.
14392
35 years
Wovex
44 weeks 0 days
26
Normal skull apperance
No choroid plexus cyst seen 
Normal 4 chamber view
Spina bifida found 
Normal 
Normal 
Hands and feet appear normal 
There is no structural defects and normal flow patterns■fetal abnormalities detected in this scan



In [30]:
doc = fitz.open(sample_pdf)
page = doc[0]

# Sorting the extracted texts
text_sorted = page.get_text("text", sort=True)
print(text_sorted)

                  Origin Hospital

      Patient ID : 14392                                                       Patient Age :  35 years
      Patient Name :  Wovex                                  GA :  44 weeks 0 days
     Gender : Female                                               BMI : 26

    Examination Findings

     Head :    Normal skull apperance

      Brain :   No choroid plexus cyst seen


     Heart :   Normal 4 chamber view

     Spine :    Spina bifida found

     Abdominal wall: Normal

     Urinary tract:   Normal

      Extremities:    Hands and feet appear normal





    Conclusion

        There is no structural defects and normal flow patternsIfetal abnormalities detected in this scan





.


In [31]:
import re

KNOWN_LABELS = [
    "Patient ID", "Patient Age", "Patient Name", "GA", "Gender", "BMI",
    "Examination Findings", "Head", "Brain", "Heart", "Spine",
    "Abdominal wall", "Urinary tract", "Extremities", "Conclusion"
]
label_alternation = "|".join(re.escape(label) for label in KNOWN_LABELS)

def extract_field(text: str, label: str) -> str:
  #ignoring the extra spaces
    pattern = rf"{re.escape(label)}\s*:\s*(.+?)(?=\s{{2,}}(?:{label_alternation})\s*:|\Z)"
    match = re.search(pattern, text)
    return match.group(1).strip() if match else None

patient_id = extract_field(text_sorted, "Patient ID")
patient_name = extract_field(text_sorted, "Patient Name")

print("Extracted Patient ID:", patient_id)
print("Extracted Patient Name:", patient_name)

Extracted Patient ID: 14392
Extracted Patient Name: Wovex


Extracts, redacts, and save PDF. Returns a metadata or None if this exact file content was already processed.

In [33]:
PDF_QUARANTINE_DIR = "/content/drive/MyDrive/Origin_Output/Quarantine_PDFs"
os.makedirs(PDF_QUARANTINE_DIR, exist_ok=True)

def deidentify_pdf(path: str):
    file_hash = get_file_hash(path)

    cur.execute("SELECT 1 FROM pdf_reports WHERE source_file_hash = %s", (file_hash,))
    if cur.fetchone() is not None:
        return {"status": "skipped"}

    doc = fitz.open(path)
    text_sorted = doc[0].get_text("text", sort=True)

    patient_id = extract_field(text_sorted, "Patient ID")
    patient_name = extract_field(text_sorted, "Patient Name")

    if not patient_id or not patient_name:

        doc.close()
        shutil.copy2(path, os.path.join(PDF_QUARANTINE_DIR, os.path.basename(path)))
        return {"status": "quarantined", "reason": "field_extraction_failed"}

    pseudo_patient_id = "ANON" + hmac_hex(str(patient_id))[:16].upper()

    for page in doc:
        for value in [patient_id, patient_name]:
            for rect in page.search_for(value):
                page.add_redact_annot(rect, fill=(0, 0, 0))
        page.apply_redactions()

    out_filename = make_output_filename(pseudo_patient_id, path)
    out_path = os.path.join(PDF_OUTPUT, out_filename)
    doc.save(out_path)
    doc.close()

    verify_doc = fitz.open(out_path)
    verify_text = " ".join(page.get_text("text", sort=True) for page in verify_doc)
    verify_doc.close()
    redaction_verified = (patient_id not in verify_text) and (patient_name not in verify_text)

    upsert_patient(pseudo_patient_id)

    insert_pdf_record({
        "pseudo_patient_id": pseudo_patient_id,
        "source_file_hash": file_hash,
        "output_filename": out_filename,
        "redaction_verified": redaction_verified,
    })

    return {"status": "processed", "redaction_verified": redaction_verified}

In [34]:
summary = {"processed": 0, "skipped": 0, "quarantined": 0, "failed": 0}
verification_warnings = []
failed_files = []
run_started = datetime.now()

for path in pdf_files:
    try:
        result = deidentify_pdf(path)
        summary[result["status"]] += 1
        if result["status"] == "processed" and not result["redaction_verified"]:
            verification_warnings.append(os.path.basename(path))
    except Exception as e:
        summary["failed"] += 1
        failed_files.append((os.path.basename(path), str(e)))

cur.execute("""
    INSERT INTO pipeline_runs (run_type, started_at, completed_at, files_processed, files_skipped, files_quarantined, files_failed)
    VALUES (\'pdf\', %s, NOW(), %s, %s, %s, %s)
""", (run_started, summary["processed"], summary["skipped"], summary["quarantined"], summary["failed"]))

print("=== PDF batch complete ===")
print(summary)
print(f"Total accounted for: {sum(summary.values())} / {len(pdf_files)}")
if verification_warnings:
    print(f"\n{len(verification_warnings)} processed but redaction not verified: {verification_warnings}")
if failed_files:
    print("\nFailed files:")
    for name, err in failed_files:
        print(f"  {name}: {err}")

=== PDF batch complete ===
{'processed': 50, 'skipped': 0, 'quarantined': 0, 'failed': 0}
Total accounted for: 50 / 50
